In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.set_default_device("cpu")


In [5]:
from transformers import T5ForConditionalGeneration, AutoTokenizer

model = T5ForConditionalGeneration.from_pretrained('fdemelo/g2p-multilingual-byt5-tiny-8l-ipa-childes')
tokenizer = AutoTokenizer.from_pretrained('fdemelo/g2p-multilingual-byt5-tiny-8l-ipa-childes')

model_inputs = tokenizer(["<fr-na>: Je t'aime."], max_length=512, padding=True, truncation=True, add_special_tokens=False, return_tensors="pt")
preds = model.generate(**model_inputs, num_beams=1, max_length=512) # We do not find beam search helpful. Greedy decoding is enough.
phones = tokenizer.batch_decode(preds.tolist(), skip_special_tokens=True)
print(phones)

['ʒə tɛm']


In [1]:
import pandas as pd
df = pd.read_csv('/Users/jungmin/Desktop/Seq2Seq project internship/g2p-byt5-french-finetuned/checkpoint-80/glaff-1.2.2.txt', sep='|', header=None, on_bad_lines='skip')

df.columns = ['Graphie', 'Code_GRACE', 'Lemme', 'Phono_API', 'Phono_SAMPA'] + [f'Col_{i}' for i in range(5, len(df.columns))]
df = df.dropna(subset =['Graphie', 'Phono_API'])
list_word = df[['Graphie', 'Phono_API']]

list_grapheme = df['Graphie'].tolist()
list_phoneme = df['Phono_API'].tolist()
list_phoneme1 = [word.replace(".", "") for word in list_phoneme]
list_word = list_word.dropna(subset=['Phono_API'])
print(list_word)

            Graphie      Phono_API
0       übercélèbre  y.bœʁ.se.lɛbʁ
1       übercélèbre  y.bœʁ.se.lɛbʁ
6              açaï          a.saj
7             açaïs          a.saj
8             aïaut           a.jo
...             ...            ...
610430        hôles             ol
610431        hales             al
610432        hâles          al;ɑl
610433        hèles             ɛl
610434        hôles             ol

[539374 rows x 2 columns]


In [2]:
merged_data = []

for graph, phono in zip(list_grapheme, list_phoneme1):
    entry = {'grapheme': graph, 'phoneme': phono}
    merged_data.append(entry)

for i, item in enumerate(merged_data):
    print(f"Grapheme: {item['grapheme']}, Phoneme: {item['phoneme']}")
    
    if i == 9: break
len(merged_data)

Grapheme: übercélèbre, Phoneme: ybœʁselɛbʁ
Grapheme: übercélèbre, Phoneme: ybœʁselɛbʁ
Grapheme: açaï, Phoneme: asaj
Grapheme: açaïs, Phoneme: asaj
Grapheme: aïaut, Phoneme: ajo
Grapheme: aïauts, Phoneme: ajo
Grapheme: AAC, Phoneme: aase
Grapheme: aède, Phoneme: aɛd
Grapheme: aèdes, Phoneme: aɛd
Grapheme: aïd, Phoneme: aid


539374

In [7]:
model_inputs = tokenizer(f"<fr-na>: {list_grapheme}",max_length=512, padding=True, truncation=True, add_special_tokens=False, return_tensors="pt")

In [8]:
preds = model.generate(**model_inputs, num_beams=1)
phones = tokenizer.batch_decode(preds.tolist(), skip_special_tokens=True)
print(phones[0:10])
result = []

['nybɛʁselɛbʁ yb']


/Users/jungmin/Desktop/Seq2Seq project internship/.venv/lib/python3.9/site-packages/transformers/generation/utils.py:1355: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


In [9]:
for word in list_grapheme:
  model_inputs = tokenizer(f"<fr-na>: {word}", max_length=512, padding=True, truncation=True, add_special_tokens=False, return_tensors="pt")
  prends = model.generate(**model_inputs, num_beams=1)
  phones = tokenizer.batch_decode(preds.tolist(), skip_special_tokens=True)
  result.append(phones)

In [10]:
result = []
for word in list_grapheme:
  inputs = tokenizer(f"<fr-na>: {word}", return_tensors="pt")
  prends = model.generate(**inputs, num_beams=1)
  phones = tokenizer.batch_decode(prends.tolist(), skip_special_tokens=True)
  result.append(phones)
listresult = [x[0] for x in result]

In [11]:
test = {'VP': 0, 'FP': 0}
i = 0
num = 0
for word in list_phoneme1:
  if word == listresult[i]:
    i += 1
    test['VP'] += 1
    num += 1
  else:
    i += 1
    test['FP'] += 1
print(num)
print(len(list_phoneme1))
print(num/len(list_phoneme1))
print(test)


163
796
0.20477386934673367
{'VP': 163, 'FP': 633}


In [ ]:
FPGrapheme = []
FPmodel = []
i = 0
num = 0
for word in list_phoneme1:
  if word == listresult[i]:
    i += 1
  else:
    FPGrapheme.append(word)
    FPmodel.append(listresult[i])
    i += 1

In [28]:
character1 = []
character2 = []
for word in FPGrapheme:
  character = list(word)
  character1.append(character)
for word in FPmodel:
  character = list(word)
  character2.append(character)

print(character1[0:5])


[['y', 'b', 'œ', 'ʁ', 's', 'e', 'l', 'ɛ', 'b', 'ʁ'], ['y', 'b', 'œ', 'ʁ', 's', 'e', 'l', 'ɛ', 'b', 'ʁ'], ['a', 's', 'a', 'j'], ['a', 'j', 'o'], ['a', 'j', 'o']]


In [49]:
total_character_number = 0
total_character_number1 = 0
for i in FPmodel:
    total_character_number += len(i)

for i in FPGrapheme:
    total_character_number1 += len(i)

print(total_character_number)
print(total_character_number1)


5433
6398


In [ ]:
VP = 0
count = 0
for i in range(len(character1)):
  for j in range(min(len(character1),len(character2))):
    try:
      if character1[i][j] ==character2[i][j]:
        VP += 1
        
      if character1[i][j] !=character2[i][j]:
        print(character1[i][j],character2[i][j])
        count += 1
    except:
      count += 1

In [ ]:
print(VP/total_character_number1)

0.6583307283526102


## fine tune 

In [23]:
df = pd.read_csv('/Users/jungmin/Desktop/Seq2Seq project internship/g2p-byt5-french-finetuned/checkpoint-80/glaff-1.2.2.txt', sep='|', header=None, on_bad_lines='skip')

df.columns = ['Graphie', 'Code_GRACE', 'Lemme', 'Phono_API', 'Phono_SAMPA'] + [f'Col_{i}' for i in range(5, len(df.columns))]
df = df[['Graphie', 'Phono_API']]
df = df.dropna(subset =['Graphie', 'Phono_API'])
df = df.drop_duplicates(subset=['Graphie'])
df = df.rename(columns={'Graphie': 'source', 'Phono_API': 'target'})
df['target'] = df['target'].str.replace('.', '')

In [7]:
from datasets import Dataset
dataset = Dataset.from_pandas(df)

dataset = dataset.train_test_split(test_size=0.2)

train_ds = dataset['train']
test_ds = dataset['test']

In [8]:
def preprocess_function(examples):
    inputs = examples["source"]
    targets = examples["target"]
    
    model_inputs = tokenizer(inputs, max_length=32, truncation=True, padding="max_length")
    
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=32, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_ds.map(preprocess_function, batched=True)
tokenized_test = test_ds.map(preprocess_function, batched=True)

Map:   0%|          | 0/431499 [00:00<?, ? examples/s]


KeyError: 'source'

In [9]:

from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./g2p-byt5-french-finetuned",
    eval_strategy="epoch",             
    learning_rate=5e-5,                      
    per_device_train_batch_size=32,          
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    save_total_limit=3,                      
    num_train_epochs=5,                      
    predict_with_generate=True,
    logging_steps=100,
)

In [10]:
from transformers import T5ForConditionalGeneration, AutoTokenizer

model_id = 'fdemelo/g2p-multilingual-byt5-tiny-8l-ipa-childes'
model = T5ForConditionalGeneration.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)


trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
)

trainer.train()

NameError: name 'tokenized_train' is not defined

In [ ]:
import torch


if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available(): 
    device = "mps"
else:
    device = "cpu"

print(f"{device}")

model.to(device)

현재 사용 중인 장치: mps


T5ForConditionalGeneration(
  (shared): Embedding(384, 1472)
  (encoder): T5Stack(
    (embed_tokens): Embedding(384, 1472)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=1472, out_features=384, bias=False)
              (k): Linear(in_features=1472, out_features=384, bias=False)
              (v): Linear(in_features=1472, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=1472, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=1472, out_features=3584, bias=False)
              (wi_1): Linear(in_features=1472, out_features=3584, bias=False)
              (w

In [14]:
def predict_ipa(word):
    
    inputs = tokenizer(word, return_tensors="pt").to(device)
    
    with torch.no_grad():
        output_tokens = model.generate(
            **inputs, 
            max_new_tokens=32, 
            do_sample=False    
        )
    
    
    ipa_result = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
    return ipa_result


test_words = ["übercélèbre", "école", "bibliothèque", "chat"]

for w in test_words:
    print(f"Word: {w} -> IPA: {predict_ipa(w)}")

Word: übercélèbre -> IPA: èbrecélèbrecélèbrecélèbr
Word: école -> IPA: ééééééééééééééé
Word: bibliothèque -> IPA: bibliothèquestibbibliothèques
Word: chat -> IPA: atat,at,at,chat,at,chat,at,chat


In [11]:
print(merged_data[0])

{'grapheme': 'übercélèbre', 'phoneme': 'ybœʁselɛbʁ'}


In [15]:
# import model
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Trainer, TrainingArguments


device = "mps" if torch.backends.mps.is_available() else "cpu"

model_name = "google/byt5-small" 
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)


In [16]:
# import dataset

raw_dataset = Dataset.from_list(merged_data)
dataset = raw_dataset.train_test_split(test_size=0.2)



In [17]:
def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["grapheme"], 
        text_target=examples["phoneme"], 
        max_length=32, 
        truncation=True, 
        padding="max_length"
    )
    return model_inputs


In [18]:
# 1. 전처리 적용 (batched=True로 속도 업!)
tokenized_ds = raw_dataset.map(preprocess_function, batched=True)

# 2. 학습용과 검증용 데이터로 나누기 (보통 9:1)
final_dataset = tokenized_ds.train_test_split(test_size=0.1)
train_dataset = final_dataset["train"]
eval_dataset = final_dataset["test"]

Map: 100%|██████████| 539374/539374 [00:41<00:00, 13112.16 examples/s]


In [19]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

In [20]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./byt5-g2p-french",
    eval_strategy="epoch",            
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=2,
    predict_with_generate=False,
    # use_mps_device=True,           
    logging_steps=100,
    fp16=False,
)


In [21]:
from transformers import Seq2SeqTrainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)


trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds, 
    eval_dataset=tokenized_ds,  
    tokenizer=tokenizer,
    data_collator=data_collator,

)

trainer.train()



/var/folders/pk/zbrrtm5j6592hj330znmxf9m0000gn/T/ipykernel_2757/1719748147.py:6: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
/Users/jungmin/Desktop/Seq2Seq project internship/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
1,0.029300,0.026230
2,0.028300,0.020193


/Users/jungmin/Desktop/Seq2Seq project internship/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/jungmin/Desktop/Seq2Seq project internship/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=134844, training_loss=0.1131986575697605, metrics={'train_runtime': 101721.5395, 'train_samples_per_second': 10.605, 'train_steps_per_second': 1.326, 'total_flos': 6.194370430766285e+16, 'train_loss': 0.1131986575697605, 'epoch': 2.0})

In [22]:
trainer.save_model("./my_g2p_model_final")
tokenizer.save_pretrained("./my_g2p_model_final")

('./my_g2p_model_final/tokenizer_config.json',
 './my_g2p_model_final/special_tokens_map.json',
 './my_g2p_model_final/added_tokens.json')

In [27]:
import torch

def predict_phoneme(word):
    inputs = tokenizer(word, return_tensors="pt").to("mps")
    

    model.eval()
    with torch.no_grad():
        output_tokens = model.generate(
            input_ids=inputs["input_ids"],
            max_length=32,
            num_beams=4,      
            early_stopping=True
        )
    
    
    phoneme = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
    return phoneme

test_words = ["je t'aime", "aimer", "Paul", "chat"]
for w in test_words:
    print(f"Graphie: {w} -> Phono: {predict_phoneme(w)}")

Graphie: je t'aime -> Phono: ʒətɛm
Graphie: aimer -> Phono: ɛme
Graphie: Paul -> Phono: pol
Graphie: chat -> Phono: ʃa
